# Poglavje 7: Gradnja klepetalnih aplikacij
## Hitri začetek z OpenAI API

Ta zvezek je prilagojen iz [Azure OpenAI vzorčne zbirke](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključuje zvezke, ki dostopajo do storitev [Azure OpenAI](aoai-assignment.ipynb).

Python OpenAI API deluje tudi z Azure OpenAI modeli, z nekaj spremembami. Več o razlikah preberite tukaj: [Kako preklapljati med OpenAI in Azure OpenAI končnimi točkami s pomočjo Pythona](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila, veliki jezikovni model poskuša napovedati besedilo, ki bo sledilo"(1). Ta "hitri začetek" zvezek bo uporabnike uvedel v osnovne koncepte LLM, osnovne zahteve paketa za začetek z AML, nežno uvod v oblikovanje pozivov in več kratkih primerov različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati storitev OpenAI](#how-to-use-openai-service)  
[1. Ustvarjanje vaše OpenAI storitve](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Pooblastila](#3.-credentials)  

[Uporabni primeri](#use-cases)    
[1. Povzetek besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Generiranje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Natančna nastavitev razvrščevalnika](#4.fine-tune-a-classifier)  

[Reference](#references)


### Ustvarite svoj prvi poziv  
Ta kratek izziv bo zagotovil osnovni uvod za pošiljanje pozivov modelu OpenAI za enostavno nalogo »povzemanje«.


**Koraki**:  
1. Namestite knjižnico OpenAI v svoje Python okolje  
2. Naložite standardne pomožne knjižnice in nastavite svoje običajne varnostne poverilnice OpenAI za storitev OpenAI, ki ste jo ustvarili  
3. Izberite model za svojo nalogo  
4. Ustvarite enostaven poziv za model  
5. Pošljite svojo zahtevo modelu API!


### 1. Namestite OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvozi pomožne knjižnice in ustvari poverilnice


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, lahko razumejo in ustvarjajo naravni jezik, na voljo pa so na platformi OpenAI z različnimi stopnjami moči in hitrosti, primernimi za različne naloge.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Oblikovanje pozivov  

"Čar velikih jezikovnih modelov je v tem, da se z usposabljanjem za minimizacijo te napake napovedi prek obsežnih količin besedila modeli na koncu naučijo konceptov, ki so koristni za te napovedi. Na primer, naučijo se konceptov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako kodirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov v velik jezikovni model je daleč najbolj vpliven besedilni poziv(1).

Velike jezikovne modele lahko spodbudimo k ustvarjanju izhodov na nekaj načinov:

Navodilo: Povejte modelu, kaj želite
Dokončanje: Spodbudite model, da dokonča začetek tega, kar želite
Demonstracija: Prikažite modelu, kaj želite, z:
Nekaj primeri v pozivu
Mnogo sto ali tisoč primerov v podatkovni zbirki za fino prilagajanje"



#### Obstajajo trije osnovni napotki za ustvarjanje pozivov:

**Pokaži in povej.** Jasno pokažite, kaj želite bodisi preko navodil, primerov ali kombinacije obojega. Če želite, da model razvrsti seznam predmetov v abecednem vrstnem redu ali klasificira odstavek po sentimentu, mu pokažite, da je to, kar želite.

**Posredujte kakovostne podatke.** Če poskušate zgraditi klasifikator ali dobiti model, da sledi vzorcu, poskrbite, da je dovolj primerov. Preverite svoje primere — model je običajno dovolj pameten, da prepozna osnovne črkovalne napake in vam da odgovor, vendar lahko tudi predpostavi, da je to namerno, kar lahko vpliva na odgovor.

**Preverite nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri ustvarjanju odgovora. Če zahtevate odgovor, kjer je samo en pravilen odgovor, potem bi te nastavili nižje. Če želite bolj raznolike odgovore, jih lahko nastavite višje. Največja napaka ljudi pri teh nastavitvah je, da domnevajo, da so to kontrolniki za "pametnost" ali "kreativnost".


Vir: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Oddaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Povzetek besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odstavka dodate 'tl;dr:'. Opazite, kako model razume, kako izvesti vrsto nalog brez dodatnih navodil. Lahko preizkušate bolj opisne pozive kot tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Nedavno delo je pokazalo znatne izboljšave pri mnogih nalogah NLP in primerjalnih testih s predhodnim učenjem na velikem korpusu besedila, ki mu sledi dodelava za specifično nalogo. Čeprav je arhitektura običajno neodvisna od naloge, ta metoda še vedno zahteva dodelavne zbirke podatkov za naloge s tisoči ali deset tisoči primerov. V nasprotju s tem ljudje običajno zmorejo opraviti novo jezikovno nalogo že le z nekaj primeri ali preprostimi navodili – nekaj, s čimer trenutni NLP sistemi še vedno večinoma težko opravljajo. Tukaj pokažemo, da povečanje jezikovnih modelov močno izboljša nalogovno neodvisno delovanje z malo primeri, včasih celo doseže konkurenčnost z dosedanjimi najboljšimi pristopi dodelave.  



Povzetek  


# Vaje za več različnih primerov uporabe  
1. Povzemanje besedila  
2. Razvrščanje besedila  
3. Generiranje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Razvrsti besedilo  
#### Izziv  
Razvrsti predmete v kategorije, ki so določene v času sklepnega sklepanja. V naslednjem primeru v poziv vključimo tako kategorije kot besedilo za razvrstitev (*playground_reference). 

Povpraševanje stranke: Pozdravljeni, pred kratkim se je na moji prenosni tipkovnici zlomila ena tipka in potreboval bom nadomestilo:

Razvrščena kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Ustvarjanje novih imen izdelkov
#### Izziv
Ustvarite imena izdelkov iz primerov besed. V pozivu vključimo informacije o izdelku, za katerega bomo ustvarili imena. Prav tako podamo podoben primer, da pokažemo vzorec, ki ga želimo prejeti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in pridobimo bolj inovativne odzive.

Opis izdelka: Domači aparat za pripravo mlečnih napitkov
Izhodiščne besede: hiter, zdrav, kompakten.
Imena izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki ustrezajo vsaki velikosti stopala.
Izhodiščne besede: prilagodljiv, ustreza, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Kuharica](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino nastavitev modelov GPT za klasifikacijo besedil](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Sodelavci
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
